In [ ]:
# Thiết lập cho Google Colab
import os
if not os.path.exists('/content/ML-labs'):
    !git clone https://github.com/pie-12/ML-labs.git /content/ML-labs

os.chdir('/content/ML-labs/')
print("✅ Cấu hình dữ liệu thành công! Thư mục làm việc hiện tại:", os.getcwd())

# Lab 2 - Kỹ thuật Giảm chiều dữ liệu (Dimensionality Reduction)

Yêu cầu:
1. **PCA, PCA hạt nhân (Kernel PCA):** Cài đặt PCA thủ công (bằng Numpy/SVD) và dùng thư viện Scikit-Learn. So sánh kết quả. Sử dụng `sklearn.decomposition.KernelPCA` trên tập dữ liệu phi tuyến (ví dụ: `make_moons`).
2. **Các Kỹ thuật Giảm chiều Khác:** Trực quan hóa bộ dữ liệu MNIST hoặc Fashion-MNIST bằng t-SNE và PCA. So sánh sự phân tách cụm.

## 1. PCA và PCA hạt nhân (Kernel PCA)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, KernelPCA
from sklearn.datasets import make_moons, fetch_openml
from sklearn.manifold import TSNE
import seaborn as sns

# Thiết lập seed để kết quả có thể lặp lại được
np.random.seed(42)

### 1.1 Cài đặt PCA thủ công bằng Numpy/SVD

Các bước thực hiện PCA thủ công:
1. Chuẩn hóa tập dữ liệu về trung tâm (trừ đi giá trị trung bình).
2. Tính toán Phân tích Giá trị Suy biến (Singular Value Decomposition - SVD).
3. Chiếu dữ liệu lên $d$ thành phần chính (principal components) đầu tiên.

In [ ]:
# Tạo một tập dữ liệu 3D ngẫu nhiên
m = 60
w1, w2 = 0.1, 0.3
noise = 0.1

angles = np.random.rand(m) * 3 * np.pi / 2 - 0.5
X = np.empty((m, 3))
X[:, 0] = np.cos(angles) + np.sin(angles)/2 + noise * np.random.randn(m) / 2
X[:, 1] = np.sin(angles) * 0.7 + noise * np.random.randn(m) / 2
X[:, 2] = X[:, 0] * w1 + X[:, 1] * w2 + noise * np.random.randn(m)

# 1. Đưa dữ liệu về trung tâm
X_centered = X - X.mean(axis=0)

# 2. Tính toán SVD
U, s, Vt = np.linalg.svd(X_centered)

# Lấy 2 thành phần chính đầu tiên
c1 = Vt.T[:, 0]
c2 = Vt.T[:, 1]

# 3. Chiếu dữ liệu xuống không gian 2D
W2 = Vt.T[:, :2]
X2D_manual = X_centered.dot(W2)

print("Kích thước dữ liệu sau khi chiếu thủ công:", X2D_manual.shape)
print("5 dòng đầu tiên của dữ liệu chiếu thủ công:\n", X2D_manual[:5])

### 1.2 Thực hiện PCA sử dụng Scikit-Learn

Bây giờ, chúng ta sẽ sử dụng thư viện `sklearn.decomposition.PCA` để thực hiện cùng một thao tác giảm chiều dữ liệu.

In [ ]:
pca = PCA(n_components=2)
X2D_sklearn = pca.fit_transform(X)

print("Kích thước dữ liệu sau khi chiếu bằng sklearn:", X2D_sklearn.shape)
print("5 dòng đầu tiên của dữ liệu chiếu bằng sklearn:\n", X2D_sklearn[:5])

### 1.3 So sánh kết quả giữa PCA thủ công và Scikit-Learn PCA

Lưu ý rằng kết quả từ Scikit-Learn PCA có thể mang dấu ngược lại so với cách tính SVD thủ công ở một số trục. Điều này là bình thường vì một trục thành phần chính có thể chỉ theo một trong hai hướng ngược nhau. Tỷ lệ phương sai được giữ lại vẫn không đổi.

In [ ]:
# So sánh hai kết quả chiếu
# Kết quả sẽ gần như giống nhau hoàn toàn (hoặc bằng nhau nhưng khác dấu)
print("Các kết quả có gần như bằng nhau không? (Bỏ qua dấu)")
print(np.allclose(np.abs(X2D_manual), np.abs(X2D_sklearn)))

### 1.4 PCA hạt nhân (Kernel PCA)

Kernel PCA được dùng để chiếu dữ liệu phi tuyến phức tạp. Chúng ta sẽ thử nghiệm trên tập dữ liệu hình mặt trăng (`make_moons`).

In [ ]:
X_moons, y_moons = make_moons(n_samples=100, noise=0.15, random_state=42)

lin_pca = KernelPCA(n_components=2, kernel="linear", fit_inverse_transform=True)
rbf_pca = KernelPCA(n_components=2, kernel="rbf", gamma=0.04, fit_inverse_transform=True)
sig_pca = KernelPCA(n_components=2, kernel="sigmoid", gamma=0.001, coef0=1, fit_inverse_transform=True)

plt.figure(figsize=(15, 4))
for subplot, pca, title in ((131, lin_pca, "Nhân tuyến tính (Linear)"), 
                            (132, rbf_pca, "Nhân RBF, $\gamma=0.04$"), 
                            (133, sig_pca, "Nhân Sigmoid, $\gamma=10^{-3}, r=1$")):
    X_reduced = pca.fit_transform(X_moons)
    plt.subplot(subplot)
    plt.title(title, fontsize=14)
    plt.scatter(X_reduced[:, 0], X_reduced[:, 1], c=y_moons, cmap=plt.cm.coolwarm)
    plt.xlabel("$z_1$", fontsize=18)
    if subplot == 131:
        plt.ylabel("$z_2$", fontsize=18, rotation=0)
    plt.grid(True)

plt.show()

## 2. Các Kỹ thuật Giảm chiều Khác: t-SNE và PCA

Chúng ta sẽ tải một phần tập dữ liệu MNIST (ví dụ: 2000 mẫu) và giảm chiều xuống không gian 2D bằng cách sử dụng cả PCA và t-SNE. Sau đó, ta sẽ vẽ đồ thị để quan sát thuật toán nào phân tách các cụm chữ số tốt hơn.

In [ ]:
# Tải một phần tập dữ liệu MNIST
mnist = fetch_openml('mnist_784', version='active', parser='auto')
X_mnist = mnist.data
y_mnist = mnist.target.astype(int)

# Chọn một tập con để tính toán nhanh hơn
np.random.seed(42)
subset_indices = np.random.choice(len(X_mnist), 2000, replace=False)
X_subset = X_mnist.iloc[subset_indices] if hasattr(X_mnist, 'iloc') else X_mnist[subset_indices]
y_subset = y_mnist.iloc[subset_indices] if hasattr(y_mnist, 'iloc') else y_mnist[subset_indices]

In [ ]:
# Áp dụng PCA
pca = PCA(n_components=2)
X_pca_reduced = pca.fit_transform(X_subset)

# Áp dụng t-SNE
tsne = TSNE(n_components=2, random_state=42)
X_tsne_reduced = tsne.fit_transform(X_subset)

In [ ]:
# Vẽ đồ thị so sánh hai phương pháp
plt.figure(figsize=(16, 6))

plt.subplot(121)
plt.scatter(X_pca_reduced[:, 0], X_pca_reduced[:, 1], c=y_subset, cmap="jet", alpha=0.5)
plt.colorbar()
plt.title("PCA - Chiếu 2D MNIST (2000 mẫu)", fontsize=14)
plt.xlabel("Thành phần chính 1")
plt.ylabel("Thành phần chính 2")
plt.grid(True)

plt.subplot(122)
plt.scatter(X_tsne_reduced[:, 0], X_tsne_reduced[:, 1], c=y_subset, cmap="jet", alpha=0.5)
plt.colorbar()
plt.title("t-SNE - Chiếu 2D MNIST (2000 mẫu)", fontsize=14)
plt.xlabel("Thành phần t-SNE 1")
plt.ylabel("Thành phần t-SNE 2")
plt.grid(True)

plt.show()

### Kết luận so sánh t-SNE và PCA

Như chúng ta có thể quan sát từ các biểu đồ trên:
- **PCA** là một kỹ thuật giảm chiều tuyến tính. Nó cố gắng bảo toàn cấu trúc tổng thể (phương sai) của dữ liệu. Tuy nhiên, các cụm chữ số khác nhau trong bộ dữ liệu MNIST bị chồng chéo rất nhiều trong phép chiếu PCA 2D.
- **t-SNE** là một kỹ thuật phi tuyến được thiết kế đặc biệt cho mục đích trực quan hóa. Nó xuất sắc trong việc bảo toàn cấu trúc cục bộ, giữ cho các điểm dữ liệu giống nhau ở gần nhau. Do đó, nó tạo ra các cụm phân tách rõ ràng và tách biệt tốt hơn nhiều cho từng lớp chữ số.